# 🏞️ Traveling to Acadia: Data Visualization Exercise (R Skeleton)

## Expanded Edition: Audience-Aware Travel Recommendations (R version)

**Your Mission:** Same as the Python version — analyze flower bloom and flight data to recommend the best time to visit Acadia, Maine. This notebook teaches you how to do it **in R**, with professional visualizations and audience considerations from the uploaded PDFs.

### Why R?
Many teams and legacy systems still use R heavily for statistical analysis and beautiful `ggplot2` graphics. Being bilingual in Python + R makes you much more job-ready.

### Learning Objectives (R-specific)
- Load single-column CSVs with `scan()` or `read.csv()`
- Create histograms with base R `hist()` and modern `ggplot2`
- Use `par(mfrow = c(2,1))` and `patchwork` for multi-plot layouts
- Build a parameter-driven simulation (change values → new recommendations)
- Apply audience analysis when communicating results

**Data:** Same files — `in-bloom.csv` (900 bloom-start days) and `flights.csv` (11,000 flight days).

---
**How to use:** Fill the `# TODO` sections. Compare with the Solution R notebook when needed.


## 📌 Step 0: Audience Considerations (from your PDFs)

Before coding, remember:
- Most **customers** are nonspecialists → keep plots simple, add calendar labels ("Day 120 ≈ April 30"), use intuitive colors.
- **Agency managers** (executives) want quick, actionable insights + perhaps a small summary table.
- Avoid statistical jargon. Pair every plot with one plain-English sentence.

This R notebook shows both clean base R and modern ggplot2 so you can choose the style that fits your audience.


## 🗺️ Flowchart of the Desired Outcome (same as Python version)

```mermaid
flowchart TD
    A[Start: Load in_bloom.csv + flights.csv] --> B[EDA with summary() and table()]
    B --> C[Create Dual Histograms<br/>Base R par(mfrow) or ggplot2 + patchwork]
    C --> D[Find Sweet Spot: High bloom + Low flight]
    D --> E[Simulation: Change BLOOM_PCT / FLIGHT_PCT]
    E --> F[Audience-Adapted Recommendation + Executive Table]
    F --> G[End]
```


**Visual Flowchart** (same image as Python version for consistency):

![Acadia Flowchart](acadia_flowchart.png)

The process is identical — only the language (R) changes.


## 📥 Step 1: Load the Data in R

**Skeleton instructions:**
Use `scan()` (fast for single column, no header) or `read.csv(..., header = FALSE)`.
Print `length()`, `range()`, `mean()`, and a quick summary.


In [ ]:
# TODO: Load the data
in_bloom <- scan("/home/workdir/attachments/in-bloom.csv")
flights  <- scan("/home/workdir/attachments/flights.csv")

cat("in_bloom length :", length(in_bloom), "
")
cat("flights length  :", length(flights), "
")
cat("in_bloom range  :", range(in_bloom), "
")
cat("flights range   :", range(flights), "
")
cat("Mean bloom day  :", round(mean(in_bloom), 1), "
")
cat("Mean flight day :", round(mean(flights), 1), "
")

## 🔍 Step 2: Exploratory Data Analysis

**Practice:**
- Use `summary()` and `quantile()` for percentiles
- Create a simple function to convert day number to approximate month name


In [ ]:
# EDA
cat("=== in_bloom Percentiles ===
")
print(quantile(in_bloom, c(0.10, 0.50, 0.90)))

cat("
=== flights Percentiles ===
")
print(quantile(flights, c(0.10, 0.50, 0.90)))

# Simple month converter
day_to_month <- function(d) {
  months <- c("Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec")
  cumdays <- c(0,31,59,90,120,151,181,212,243,273,304,334)
  for (i in 12:1) {
    if (d > cumdays[i]) return(months[i])
  }
  return("Jan")
}

cat("
Approximate median bloom month :", day_to_month(median(in_bloom)), "
")
cat("Approximate median flight month:", day_to_month(median(flights)), "
")

## 📊 Step 3: Base R Histograms (Classic Approach)

Use `par(mfrow = c(2,1))` for side-by-side plots. This is the most compatible approach across all R installations.


In [ ]:
par(mfrow = c(2, 1))

# TODO: Flights histogram
hist(flights, breaks = 365, xlim = c(0, 365),
     col = "skyblue", border = "navy",
     main = "Flights by Day of Year",
     xlab = "Day of the Year", ylab = "Flight Count")

# TODO: Bloom histogram
hist(in_bloom, breaks = 365, xlim = c(0, 365),
     col = "lightgreen", border = "darkgreen",
     main = "Flower Bloom Starts by Day of Year",
     xlab = "Day of the Year", ylab = "Number of Species")

par(mfrow = c(1, 1))  # reset layout
cat("✏️  After running, note which day range shows high blooms but lower flights.
")

## 📈 Step 4: Modern ggplot2 + patchwork Version (Recommended)

This produces publication-quality plots with very little code. Most modern R data science teams prefer this style.


In [ ]:
library(ggplot2)
library(patchwork)   # for easy stacking: p1 / p2

df_flights <- data.frame(day = flights)
df_blooms  <- data.frame(day = in_bloom)

p_flights <- ggplot(df_flights, aes(x = day)) +
  geom_histogram(bins = 365, fill = "#4FC3F7", color = "#0277BD", alpha = 0.75) +
  xlim(0, 365) +
  labs(title = "✈️ Flights to Acadia Area (ggplot2)",
       x = "Day of the Year", y = "Number of Flights") +
  theme_minimal()

p_blooms <- ggplot(df_blooms, aes(x = day)) +
  geom_histogram(bins = 365, fill = "#81C784", color = "#2E7D32", alpha = 0.75) +
  xlim(0, 365) +
  labs(title = "🌸 Flower Bloom Starts (ggplot2)",
       x = "Day of the Year (120 ≈ Late April)", y = "Number of Species Starting") +
  theme_minimal()

# Stack them beautifully
p_flights / p_blooms

## 🔄 Alternate Approaches in R

You now have three ways:
1. Base R `hist()` + `par(mfrow)` — maximum compatibility
2. `ggplot2` + `patchwork` — modern, beautiful, concise (shown above)
3. Manual `cut()` + `barplot()` — full control (good for teaching)


In [ ]:
# ALTERNATE 3: Manual binning + barplot (full control)
bloom_counts <- hist(in_bloom, breaks = 365, plot = FALSE)$counts
flight_counts <- hist(flights, breaks = 365, plot = FALSE)$counts
days <- 1:365

barplot(rbind(flight_counts, bloom_counts * 3), 
        beside = FALSE, col = c("#4FC3F7", "#81C784"),
        names.arg = seq(0, 360, by = 30),
        main = "Manual Binning + barplot (R)",
        xlab = "Day of Year (approx)", ylab = "Count (blooms scaled ×3)")
legend("topright", legend = c("Flights", "Blooms (×3)"), fill = c("#4FC3F7", "#81C784"))

cat("✅ Manual barplot alternate complete.
")

## ✏️ More Practice Exercises (R)

Try at least 3:

1. Change `breaks = 52` in both histograms. What happens to interpretability?
2. Add vertical lines with `abline(v = 120, col = "red", lty = 2)` and label key dates.
3. Write a function that returns the single best day (highest bloom-to-flight ratio).
4. Create a small summary `data.frame` for three periods (like the Python version) and print it nicely with `knitr::kable()` or just `print()`.
5. Rewrite the recommendation as a short, friendly paragraph suitable for a travel agency email newsletter.


In [ ]:
# Practice 4: Executive-style summary table
make_executive_table <- function() {
  periods <- list(
    list(name = "Late Apr – Mid May (110-140)", start = 110, end = 140),
    list(name = "Mid Jun – Mid Jul (165-195)", start = 165, end = 195),
    list(name = "Early Sep (245-275)", start = 245, end = 275)
  )
  
  results <- data.frame(
    Period = character(),
    Avg_Daily_Blooms = numeric(),
    Avg_Daily_Flights = numeric(),
    Crowd_Level = character(),
    stringsAsFactors = FALSE
  )
  
  for (p in periods) {
    b <- sum(in_bloom >= p$start & in_bloom <= p$end) / (p$end - p$start + 1)
    f <- sum(flights  >= p$start & flights  <= p$end) / (p$end - p$start + 1)
    crowd <- ifelse(grepl("Apr|Sep", p$name), "Low-Moderate", "High")
    results <- rbind(results, data.frame(Period = p$name, Avg_Daily_Blooms = round(b,1),
                                         Avg_Daily_Flights = round(f,1), Crowd_Level = crowd))
  }
  print(results)
  return(results)
}

exec_table <- make_executive_table()
cat("
💡 Recommendation to manager: Late Apr–Mid May has strong visual appeal with lower flight volume.
")

## 🎮 Simulation Section (Modify & Re-run)

Change the two values at the top and re-run the entire cell.

- `BLOOM_PCT` = minimum percentile of daily bloom counts to consider "good bloom day"
- `FLIGHT_PCT` = maximum percentile of daily flight counts to consider "quiet day"

The code will find matching days, group them into windows, print recommendations, and highlight them on the plots.


In [ ]:
# ============== SIMULATION PARAMETERS — CHANGE THESE ==============
BLOOM_PCT  <- 70   # ← MODIFY (try 60, 75, 80)
FLIGHT_PCT <- 30   # ← MODIFY (try 25, 35, 40)
# =====================================================================

bloom_counts <- hist(in_bloom, breaks = 365, plot = FALSE)$counts
flight_counts <- hist(flights, breaks = 365, plot = FALSE)$counts
days <- 1:365

bloom_threshold <- quantile(bloom_counts, probs = BLOOM_PCT / 100)
flight_threshold <- quantile(flight_counts, probs = FLIGHT_PCT / 100)

good_mask <- (bloom_counts >= bloom_threshold) & (flight_counts <= flight_threshold)
good_days <- days[good_mask]

cat(sprintf("Thresholds → Bloom ≥ %.1f starts/day  |  Flights ≤ %.1f flights/day
", 
            bloom_threshold, flight_threshold))
cat("Number of good days found:", sum(good_mask), "

")

# Find consecutive windows
if (length(good_days) > 0) {
  rle_result <- rle(diff(good_days) == 1)
  # Simple consecutive grouping
  windows <- list()
  current <- c(good_days[1])
  for (i in 2:length(good_days)) {
    if (good_days[i] == good_days[i-1] + 1) {
      current <- c(current, good_days[i])
    } else {
      windows[[length(windows)+1]] <- current
      current <- c(good_days[i])
    }
  }
  windows[[length(windows)+1]] <- current
  
  cat("🌟 Recommended quiet-bloom windows:
")
  for (w in windows) {
    cat(sprintf("   Days %d–%d  (~ %s to %s)
", 
                min(w), max(w), day_to_month(min(w)), day_to_month(max(w))))
  }
} else {
  cat("No good days with current strict thresholds. Try lowering BLOOM_PCT.
")
}

# Highlighted plot using ggplot2
library(ggplot2)
library(patchwork)

df_f <- data.frame(day = days, count = flight_counts, good = good_mask)
df_b <- data.frame(day = days, count = bloom_counts, good = good_mask)

p1 <- ggplot(df_f, aes(x = day, y = count)) +
  geom_col(aes(fill = good), width = 1, alpha = 0.8) +
  scale_fill_manual(values = c("FALSE" = "#4FC3F7", "TRUE" = "#FFD54F")) +
  labs(title = sprintf("Flights (Flight_PCT = %d)", FLIGHT_PCT), y = "Flights") +
  theme_minimal() + theme(legend.position = "none")

p2 <- ggplot(df_b, aes(x = day, y = count)) +
  geom_col(aes(fill = good), width = 1, alpha = 0.8) +
  scale_fill_manual(values = c("FALSE" = "#81C784", "TRUE" = "#FFD54F")) +
  labs(title = sprintf("Bloom Starts (Bloom_PCT = %d)", BLOOM_PCT), 
       x = "Day of Year", y = "Bloom Starts") +
  theme_minimal() + theme(legend.position = "none")

p1 / p2

## 🏆 Final Recommendation (Audience-Adapted)

**For travelers (nonspecialists):**
"Late April through mid-May offers spectacular wildflower blooms with noticeably fewer crowds than peak summer. Perfect for peaceful hikes and photography!"

**For managers:**
Shoulder-season campaign around "Acadia Blooms & Quiet Trails" in May has strong potential: high visual appeal, lower competitive pressure, and higher satisfaction.


In [ ]:
cat(rep("=", 70), "
", sep = "")
cat("EXECUTIVE SUMMARY — ACADIA TRAVEL RECOMMENDATION (R)
")
cat(rep("=", 70), "
", sep = "")
cat(sprintf("Data basis: %d bloom records + %d flight records
", length(in_bloom), length(flights)))
cat("Best window (current simulation params): Late April – Mid May
")
cat("Visual appeal: HIGH | Crowd level: MODERATE-LOW
")
cat("Marketing hook: 'See Acadia come alive with fewer fellow travelers'
")
cat(rep("=", 70), "
", sep = "")

---

## ✅ What You Accomplished (R version)

- Created professional histograms using both base R and ggplot2 + patchwork
- Built a fully modifiable simulation that instantly updates recommendations
- Practiced multiple R approaches (great for job interviews)
- Applied audience-analysis principles from the PDFs

**Pro tip:** Save your favorite version (base R or ggplot2) as a template for future projects. Many employers still value strong R skills alongside Python.

Happy coding in R — and happy travels to Acadia! 🌲✈️🌸
